# 04 - Regularization, other models and ensembles

**Starting point:** v2 family rule scored **0.799** on Kaggle. v3 (ticket groups) changes 1 passenger.

What's tried here:
1. **Regularization** of logistic regression
2. **Other models:** SVM, random forest, gradient boosting, XGBoost
3. **Ensembles:** soft voting (average the models' answers) and stacking (a second model learns how to combine them)
4. **Hybrid:** family rule for passengers with a known family, ensemble for the rest

**Result in short:** only the hybrid helps, and only for **boys with no known family**. Output: `submission_v4.csv`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                              StackingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_predict, cross_val_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

train = pd.read_csv("titanic/train.csv")
test = pd.read_csv("titanic/test.csv")
df = pd.concat([train, test], ignore_index=True)
is_train = df.Survived.notna()

## Features
Same ideas as notebook 01, plus *people on the ticket* and *fare per person*.
Fill values come from train only.

In [2]:
df["Title"] = df.Name.str.extract(r",\s*([^\.]+)\.")[0]
t = df.Title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df["TitleGroup"] = t.where(t.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

df["AgeFilled"] = df.Age.fillna(df.TitleGroup.map(df[is_train].groupby("TitleGroup").Age.median()))
df["IsFemale"] = (df.Sex == "female").astype(int)
df["FamilySize"] = df.SibSp + df.Parch + 1
df["TicketCount"] = df.groupby("Ticket").Ticket.transform("size")
df["FarePerPerson"] = df.Fare.fillna(df[is_train].Fare.median()) / df.TicketCount
df["HasCabin"] = df.Cabin.notna().astype(int)
for p in "CQS":
    df[f"From{p}"] = (df.Embarked.fillna("S") == p).astype(int)
for x in ["Mr", "Mrs", "Miss", "Master", "Rare"]:
    df[f"Title_{x}"] = (df.TitleGroup == x).astype(int)

FEATURES = ["Pclass", "IsFemale", "AgeFilled", "FamilySize", "TicketCount", "FarePerPerson", "HasCabin",
            "FromC", "FromQ", "FromS", "Title_Mr", "Title_Mrs", "Title_Miss", "Title_Master", "Title_Rare"]
X = df.loc[is_train, FEATURES]
y = df.loc[is_train, "Survived"].astype(int)

# 5-fold cross-validation, repeated 3 times with different splits, for steadier numbers
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
print(f"Gender rule on train: {(X.IsFemale == y).mean():.3f}")

Gender rule on train: 0.787


## 1. Regularization
`C` controls how much the model is held back. Small C = strong regularization (simpler model).

In [3]:
for C in [0.01, 0.03, 0.1, 0.3, 1, 10]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
    print(f"C = {C:<5}: {cross_val_score(m, X, y, cv=cv).mean():.3f}")

C = 0.01 : 0.830


C = 0.03 : 0.833


C = 0.1  : 0.831


C = 0.3  : 0.830


C = 1    : 0.827


C = 10   : 0.828


**Flat: 0.827–0.833.** Regularization makes almost no difference here. The model isn't overfitting much. It just lacks information.

## 2. Other models

In [4]:
models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=2000)),
    "SVM": make_pipeline(StandardScaler(), SVC(C=1, probability=True, random_state=0)),
    "Random forest": RandomForestClassifier(500, max_depth=5, min_samples_leaf=3, random_state=0),
    "Gradient boosting": GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=0),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, random_state=0, verbosity=0),
}
scores = {name: cross_val_score(m, X, y, cv=cv).mean() for name, m in models.items()}
pd.Series(scores, name="CV accuracy").round(3)

Logistic regression    0.830
SVM                    0.831
Random forest          0.830
Gradient boosting      0.833
XGBoost                0.837
Name: CV accuracy, dtype: float64

## 3. Ensembles

In [5]:
estimators = list(models.items())
vote = VotingClassifier(estimators, voting="soft")
stack = StackingClassifier(estimators, final_estimator=LogisticRegression(max_iter=2000), cv=5)

print(f"Soft voting: {cross_val_score(vote, X, y, cv=cv).mean():.3f}")
print(f"Stacking   : {cross_val_score(stack, X, y, cv=cv).mean():.3f}")

Soft voting: 0.836


Stacking   : 0.836


**All models and ensembles land at 0.83–0.84.** That's the same as v1, which scored **0.76** on Kaggle.
So none of these should be submitted on their own. They lose to the family rule (0.799).

## 4. Hybrid: family rule + ensemble
Family rule for passengers whose family is known. Ensemble only for the rest.
Groups are built exactly like notebook 03 (family + ticket).

In [6]:
df["IsWomanOrChild"] = (df.Sex == "female") | (df.Title == "Master")
df["Surname"] = df.Name.str.split(",").str[0]
df["GroupId"] = (df.Surname + "-" + df.Pclass.astype(str) + "-" + df.Ticket.str[:-1]
                 + "-" + df.Fare.astype(str) + "-" + df.Embarked.fillna(""))
df.loc[~df.IsWomanOrChild, "GroupId"] = "none"
df.loc[df.groupby("GroupId").GroupId.transform("size") < 2, "GroupId"] = "none"
for i in df.index[(df.GroupId == "none") & df.IsWomanOrChild]:
    match = df.loc[(df.GroupId != "none") & (df.Ticket == df.at[i, "Ticket"]), "GroupId"]
    if len(match):
        df.at[i, "GroupId"] = match.iloc[0]

def family_rule(rows, labelled, leave_one_out=False):
    """Returns predictions, and which passengers had a known family."""
    stats = labelled[labelled.GroupId != "none"].groupby("GroupId").Survived.agg(["sum", "count"])
    pred = (rows.Sex == "female").astype(int)
    known = pd.Series(False, index=rows.index)
    for i, r in rows.iterrows():
        if r.GroupId not in stats.index:
            continue
        s, c = stats.loc[r.GroupId]
        if leave_one_out:
            s, c = s - r.Survived, c - 1
        if c == 0:
            continue
        known[i] = True
        if r.Title == "Master" and s == c:
            pred[i] = 1
        elif r.Sex == "female" and s == 0:
            pred[i] = 0
    return pred, known

trn, tst = df[is_train], df[~is_train]
rule_pred, known = family_rule(trn, trn, leave_one_out=True)
print(f"Family rule on train: {(rule_pred == trn.Survived).mean():.3f}")
print("Passengers with a known family:", known.sum(), "of", len(trn))

Family rule on train: 0.846
Passengers with a known family: 129 of 891


### Which passengers should the ensemble handle?
For each group of passengers with no known family, let the ensemble replace the rule.
Count how many answers it flips, and how many flips are right.
Each passenger is scored by a model that never saw them (out-of-fold).

In [7]:
no_family = ~known
segments = {
    "Everyone with no known family": no_family,
    "Women, no known family": no_family & (trn.Sex == "female"),
    "Boys, no known family": no_family & (trn.Title == "Master"),
    "Adult men": no_family & (trn.Sex == "male") & (trn.Title != "Master"),
    "1st class adult men": no_family & (trn.Sex == "male") & (trn.Title != "Master") & (trn.Pclass == 1),
}
rows = []
for model_name, m in [("Voting", vote), ("Stacking", stack), ("XGBoost", models["XGBoost"])]:
    oof = pd.Series(cross_val_predict(m, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=1)), index=trn.index)
    for seg, mask in segments.items():
        flip = mask & (oof != rule_pred)
        right = (oof[flip] == trn.Survived[flip]).sum()
        rows.append({"Model": model_name, "Passengers": seg, "Flips": flip.sum(),
                     "Right": right, "Wrong": flip.sum() - right, "Net gain": 2 * right - flip.sum()})
pd.DataFrame(rows).set_index(["Model", "Passengers"])

Flips  Right  Wrong  Net gain
Model    Passengers                                                  
Voting   Everyone with no known family     32     16     16         0
         Women, no known family            16      6     10        -4
         Boys, no known family              5      5      0         5
         Adult men                         11      5      6        -1
         1st class adult men               11      5      6        -1
Stacking Everyone with no known family     38     20     18         2
         Women, no known family            15      6      9        -3
         Boys, no known family              5      5      0         5
         Adult men                         18      9      9         0
         1st class adult men               18      9      9         0
XGBoost  Everyone with no known family     67     32     35        -3
         Women, no known family            27     10     17        -7
         Boys, no known family              5      5      0         5
         Adult men                         35     17     18        -1
         1st class adult men               34     16     18        -2

### Stability check: does it hold with different splits?
Run the voting ensemble 5 times with different cross-validation splits.
(This reuses the same 891 passengers, so it's a stability check, not proof.)

In [8]:
boys = no_family & (trn.Title == "Master")
adults = no_family & (trn.Sex == "male") & (trn.Title != "Master")
rows = []
for seed in range(5):
    for model_name, m in [("Voting", vote), ("Stacking", stack)]:
        oof = pd.Series(cross_val_predict(m, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed)), index=trn.index)
        row = {"Seed": seed, "Model": model_name}
        for seg, mask in [("Boys", boys), ("Adult men", adults)]:
            flip = mask & (oof != rule_pred)
            right = (oof[flip] == trn.Survived[flip]).sum()
            row[f"{seg}: right"] = right
            row[f"{seg}: wrong"] = flip.sum() - right
        rows.append(row)
pd.DataFrame(rows).set_index(["Model", "Seed"]).sort_index()

Boys: right  Boys: wrong  Adult men: right  Adult men: wrong
Model    Seed                                                              
Stacking 0               5            0                12                10
         1               5            0                 9                 9
         2               5            0                12                18
         3               5            0                12                14
         4               5            0                11                12
Voting   0               5            0                 6                 9
         1               5            0                 5                 6
         2               5            0                 8                10
         3               5            0                 5                 7
         4               5            0                 4                 8

**What this shows**
- **Boys with no known family:** the ensemble is right on every flip, in every run. **Kept.**
- **Adult men:** voting gets more flips wrong than right in every run. Stacking is about even (56 right, 63 wrong over 5 runs). **Not used.**
- **Women with no known family:** more wrong than right. **Not used.**
- **Voting vs stacking:** stacking swings more between runs, so voting is used.

**Caution:** there are only 5 such boys in train, and this split was chosen after seeing the results.
Boy vs man is the same split the family rule already uses, so no further slicing was done.

## Build v4
- Start from the v3 family rule.
- For **boys with no known family**, use the voting ensemble's answer.

In [9]:
vote.fit(X, y)
test_pred, test_known = family_rule(tst, trn)
ens_pred = pd.Series(vote.predict(tst[FEATURES]), index=tst.index)

use_ensemble = ~test_known & (tst.Title == "Master")
final = test_pred.copy()
final[use_ensemble] = ens_pred[use_ensemble]

sub = pd.DataFrame({"PassengerId": tst.PassengerId.astype(int), "Survived": final.astype(int).values})
sub.to_csv("submission_v4.csv", index=False)
print(len(sub), "rows,", f"{sub.Survived.mean():.0%} predicted survived")

418 rows, 37% predicted survived


## What changed vs v3?

In [10]:
v3 = pd.read_csv("submission_v3.csv")
changed = sub.Survived.values != v3.Survived.values
print("Test passengers changed vs v3:", changed.sum())
tst[changed][["PassengerId", "Name", "Pclass", "Age", "SibSp", "Parch"]].assign(v3=v3.Survived.values[changed], v4=sub.Survived.values[changed])

Test passengers changed vs v3: 5


,PassengerId,Name,Pclass,Age,SibSp,Parch,v3,v4
912,913,"Olsen, Master. Artur Karl",3,9.00,0,1,0,1
1083,1084,"van Billiard, Master. Walter John",3,11.50,1,1,0,1
1172,1173,"Peacock, Master. Alfred Edward",3,0.75,1,1,0,1
1230,1231,"Betros, Master. Seman",3,NaN,0,0,0,1
1235,1236,"van Billiard, Master. James William",3,NaN,1,1,0,1


## Summary
| Tried | Result |
|---|---|
| Regularization (C 0.01–10) | No difference |
| SVM, random forest, gradient boosting, XGBoost | All about 0.83 on train, same as v1 (Kaggle 0.76) |
| Voting and stacking ensembles | 0.836 on train, no better alone |
| Ensemble for adult men / women with no family | More wrong than right, dropped |
| **Ensemble for boys with no known family** | **Right every time on train, kept** |

- Each test passenger is worth 0.24% on Kaggle.
- Expect a small change: a few passengers either way.
- This is close to the limit for honest approaches. Stopping after v4 is recommended.